## This code loads the npz files for the finetuning datasets from s3 and checks if the files are the same and not corrupted

In [1]:
import numpy as np
import pandas as pd
import boto3
import s3fs
import io
import os

In [2]:
# Define the bucket name and prefixes 
datasets = ['ptbxl', 'cpsc2018', 'cs', 'csn']

# Output key
bucket_out = 'walkky-ml'

# Output keys for two independent runs on single-labeled datasets excluding multiple-labeled records using label encoding only
#prefix_out_1 = "aruna-files/vqvae_12lead_d384_bg_cnn_4h5/vqvae/bert_finetuning"
#prefix_out_2 = "aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning"

# Check multiple-labeled datasets; 1 gave high test AUROC and 2 gave low test AUROC on cpsc2018 dataset at use_frac 1

# Output keys for two independent runs on datasets incuding multiple-labeled records using multi-hot-encoding
prefix_out_1 = "aruna-files/vqvae/bert_finetuning"
prefix_out_2 = "aruna-files/vqvae/bert_finetuning_v5_WQV/bert_finetuning"

# Initialize the S3 client
s3_client = boto3.client('s3')
s3_fs = s3fs.S3FileSystem()


In [3]:
def _signals_npz_key(prefix_out:str, dataset_name: str) -> str:
    return f"{prefix_out}/{dataset_name}_signals.npz"

In [4]:
# Read results from npz files
def read_ecg_signals_from_s3(prefix_out: str, dataset_name: str) -> dict:
    """
    Returns
    -------
    dict with:
        signals    : list[np.ndarray]  — one (length_i, 12) array per record,
                                          padding trimmed off
        labels     : label per record
        ids        : (N,) array
        strat_fold : (N,) int array 
    """
    s3_path = f"s3://{bucket_out}/{_signals_npz_key(prefix_out, dataset_name)}"
    with s3_fs.open(s3_path, "rb") as f:
        data = np.load(io.BytesIO(f.read()), allow_pickle=True)

        padded  = data["ecg_signals"]   # (N, max_len, n_leads)
        lengths = data["lengths"]       # (N,)
        signals = [padded[i, :lengths[i], :] for i in range(len(padded))]

        result = {
            "signals": signals,
            "labels":  data["labels"],
            "ids":     data["ids"],
        }
        if "strat_fold" in data:
            result["strat_fold"] = data["strat_fold"]

    print(f"[{dataset_name}] loaded {len(signals):,} signals ← {s3_path}")
    return result

In [5]:
def checkdataset(prefix_out_1: str, prefix_out_2: str, datasetname: str) -> tuple[bool, bool, bool, bool, int, int]:
    """ Check if the two dataset files stored at the prefix out 1 and 2 locations are identical """

    print(f"Comparing {datasetname} from {prefix_out_1} and {prefix_out_2}")
    file1 = read_ecg_signals_from_s3(prefix_out_1, datasetname)
    file2 = read_ecg_signals_from_s3(prefix_out_2, datasetname)
    
    # Check if both files' signals are the same
    signalcheck = all(np.array_equal(a, b, equal_nan=True) for a, b in zip(file1["signals"], file2["signals"]))
    # Check if both files' labels are the same
    labelcheck = np.array_equal(file1["labels"], file2["labels"], equal_nan=True)
    # Check if both files' ids are the same
    idcheck = np.array_equal(file1["ids"], file2["ids"])
    # Check if both files' strat_fold are the same
    stratfoldcheck = np.array_equal(file1.get("strat_fold"), file2.get("strat_fold"))
    # How many unique patients, total records
    unique_ids = len(np.unique(file1["ids"]))
    total_records = len(file1["ids"])

    del file1, file2
    return signalcheck, labelcheck, idcheck, stratfoldcheck, unique_ids, total_records


In [6]:
# Check if sentence shards are the same

def _load_shard(prefix_out, dataset_name, split_name, use_frac, shard_idx, num_shards=4):
    use_frac_str = f"{use_frac:.2f}" if split_name == "train" else "full"
    key = f"{prefix_out}/sentences/{dataset_name}_{split_name}_{use_frac_str}_shard_{shard_idx:04d}_of_{num_shards:04d}.npz"
    with s3_fs.open(f"s3://{bucket_out}/{key}", "rb") as f:
        return dict(np.load(io.BytesIO(f.read()), allow_pickle=True))

def check_sentences(prefix_1, prefix_2, dataset_name, split_name, use_frac, num_shards=4):
    for si in range(num_shards):
        d1 = _load_shard(prefix_1, dataset_name, split_name, use_frac, si, num_shards)
        d2 = _load_shard(prefix_2, dataset_name, split_name, use_frac, si, num_shards)
        tok_eq = np.array_equal(d1["sentence_tokens"], d2["sentence_tokens"])
        lab_eq = np.array_equal(d1["labels"], d2["labels"])
        id_eq  = np.array_equal(d1["ecg_idxs"], d2["ecg_idxs"])
        #emb_eq = np.array_equal(d1["xresnet_embeddings"], d2["xresnet_embeddings"])
        emb_eq = np.allclose(d1["xresnet_embeddings"], d2["xresnet_embeddings"], atol=1e-3)
        #embdiff = d1["xresnet_embeddings"] - d2["xresnet_embeddings"]
        n_diff = 0 if tok_eq else int((d1["sentence_tokens"] != d2["sentence_tokens"]).sum())
        #print(f"shard {si}: tokens_equal={tok_eq} (n_diff_values={n_diff}) labels_equal={lab_eq} ids_equal={id_eq} xresnet_embs_equal={emb_eq} {embdiff}")
        print(f"shard {si}: tokens_equal={tok_eq} (n_diff_values={n_diff}) labels_equal={lab_eq} ids_equal={id_eq} xresnet_embs_equal={emb_eq}")

In [18]:
for datasetname in datasets:
    print(f"{datasetname}: {checkdataset(prefix_out_1, prefix_out_2, datasetname)}") 

Comparing ptbxl from aruna-files/vqvae/bert_finetuning and aruna-files/vqvae/bert_finetuning_v5_WQV/bert_finetuning
[ptbxl] loaded 21,388 signals ← s3://walkky-ml/aruna-files/vqvae/bert_finetuning/ptbxl_signals.npz
[ptbxl] loaded 21,388 signals ← s3://walkky-ml/aruna-files/vqvae/bert_finetuning_v5_WQV/bert_finetuning/ptbxl_signals.npz
ptbxl: (True, True, True, True, 18617, 21388)
Comparing cpsc2018 from aruna-files/vqvae/bert_finetuning and aruna-files/vqvae/bert_finetuning_v5_WQV/bert_finetuning
[cpsc2018] loaded 6,877 signals ← s3://walkky-ml/aruna-files/vqvae/bert_finetuning/cpsc2018_signals.npz
[cpsc2018] loaded 6,877 signals ← s3://walkky-ml/aruna-files/vqvae/bert_finetuning_v5_WQV/bert_finetuning/cpsc2018_signals.npz
cpsc2018: (True, True, True, True, 6877, 6877)
Comparing cs from aruna-files/vqvae/bert_finetuning and aruna-files/vqvae/bert_finetuning_v5_WQV/bert_finetuning
[cs] loaded 10,646 signals ← s3://walkky-ml/aruna-files/vqvae/bert_finetuning/cs_signals.npz
[cs] loaded 10

In [19]:
dataset_name = "cpsc2018"
use_frac = 1.0
for split_name in ["train", "val", "test"]:
    check_sentences(prefix_out_1, prefix_out_2, dataset_name, split_name, use_frac, num_shards=4)

shard 0: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 1: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 2: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 3: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 0: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 1: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 2: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 3: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 0: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 1: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 2: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True
shard 3: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True


In [20]:
# Check cs
file1 = read_ecg_signals_from_s3(prefix_out_1, "cs")
file2 = read_ecg_signals_from_s3(prefix_out_2, "cs")


[cs] loaded 10,646 signals ← s3://walkky-ml/aruna-files/vqvae/bert_finetuning/cs_signals.npz
[cs] loaded 10,646 signals ← s3://walkky-ml/aruna-files/vqvae/bert_finetuning_v5_WQV/bert_finetuning/cs_signals.npz


In [21]:
ct = 0
for i in range(len(file1['signals'])):
    if not np.allclose(file1['signals'][i], file2['signals'][i], atol=1e-6, rtol=1e-6, equal_nan=True):
        ct += 1
print(f"Number of  different records: {ct}")

Number of  different records: 0


In [22]:
# Check cpsc

datasetname = "cpsc2018"
print(f"{datasetname}: {checkdataset(prefix_out_1, prefix_out_2, datasetname)}")

Comparing cpsc2018 from aruna-files/vqvae/bert_finetuning and aruna-files/vqvae/bert_finetuning_v5_WQV/bert_finetuning
[cpsc2018] loaded 6,877 signals ← s3://walkky-ml/aruna-files/vqvae/bert_finetuning/cpsc2018_signals.npz
[cpsc2018] loaded 6,877 signals ← s3://walkky-ml/aruna-files/vqvae/bert_finetuning_v5_WQV/bert_finetuning/cpsc2018_signals.npz
cpsc2018: (True, True, True, True, 6877, 6877)


In [23]:
# Check single-labeled datasets; 1 gave high test AUROC and 2 gave low test AUROC at use_frac 1
datasets = ["ptbxl", "cpsc2018", "cs"]

# Output keys for two independent runs on single-labeled datasets excluding multiple-labeled records using label encoding only
prefix_out_1 = "aruna-files/vqvae_12lead_d384_bg_cnn_4h5/vqvae/bert_finetuning"
prefix_out_2 = "aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning"

for datasetname in datasets:
    print(f"{datasetname}: {checkdataset(prefix_out_1, prefix_out_2, datasetname)}") 

Comparing ptbxl from aruna-files/vqvae_12lead_d384_bg_cnn_4h5/vqvae/bert_finetuning and aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning
[ptbxl] loaded 16,244 signals ← s3://walkky-ml/aruna-files/vqvae_12lead_d384_bg_cnn_4h5/vqvae/bert_finetuning/ptbxl_signals.npz
[ptbxl] loaded 16,244 signals ← s3://walkky-ml/aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning/ptbxl_signals.npz
ptbxl: (True, True, True, True, 14627, 16244)
Comparing cpsc2018 from aruna-files/vqvae_12lead_d384_bg_cnn_4h5/vqvae/bert_finetuning and aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning
[cpsc2018] loaded 6,401 signals ← s3://walkky-ml/aruna-files/vqvae_12lead_d384_bg_cnn_4h5/vqvae/bert_finetuning/cpsc2018_signals.npz
[cpsc2018] loaded 6,401 signals ← s3://walkky-ml/aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning/cpsc2018_signals.npz
cpsc2018: (True, True, True, True, 6401, 6401)
Comparing cs from aruna-files/vqvae_12lead_d384_bg_cnn_4h5/vqvae/bert_fi

In [8]:
# check cpsc2018, ptbxl, csn, cs sentences, 1st run and rerun with xrestnet embeddings - these are not the same within tol of 1e-3!!

# Output key and prefixes
bucket_out = 'walkky-ml'
prefix_out_1 = "aruna-files/vqvae/bert_finetuning/sentences_1strun"
prefix_out_2 = "aruna-files/vqvae/bert_finetuning/sentences_rerun"

# Initialize the S3 client
s3_client = boto3.client('s3')
s3_fs = s3fs.S3FileSystem()

for dataset_name in ["cpsc2018", "ptbxl", "csn", "cs"]:
    use_frac = 1.0
    for split_name in ["train", "val", "test"]:
        print(f"dataset {dataset_name}, split {split_name}")
        check_sentences(prefix_out_1, prefix_out_2, dataset_name, split_name, use_frac, num_shards=4)

dataset cpsc2018, split train
shard 0: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 1: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 2: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 3: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
dataset cpsc2018, split val
shard 0: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 1: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 2: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 3: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
dataset cpsc2018, split test
shard 0: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_emb

In [10]:
# check cpsc2018, ptbxl, csn, cs sentences, 1st run and rerun with xrestnet embeddings - these are not the same within tol of 1e-3!!

# Output key and prefixes
bucket_out = 'walkky-ml'
prefix_out_1 = "aruna-files/vqvae/bert_finetuning/sentences_1strun"
prefix_out_2 = "aruna-files/vqvae/bert_finetuning/sentences_rerun"

# Initialize the S3 client
s3_client = boto3.client('s3')
s3_fs = s3fs.S3FileSystem()

for dataset_name in ["cpsc2018", "ptbxl", "csn", "cs"]:
    split_name = "train"
    for use_frac in [0.01, 0.1]:
        print(f"dataset {dataset_name}, use_frac {use_frac}")
        check_sentences(prefix_out_1, prefix_out_2, dataset_name, split_name, use_frac, num_shards=4)

dataset cpsc2018, use_frac 0.01
shard 0: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 1: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 2: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 3: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
dataset cpsc2018, use_frac 0.1
shard 0: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 1: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 2: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
shard 3: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresnet_embs_equal=False
dataset ptbxl, use_frac 0.01
shard 0: tokens_equal=True (n_diff_values=0) labels_equal=True ids_equal=True xresne

In [28]:
import boto3, hashlib

# Compare xresnet1d101 models 

model_key = "aruna-files/vqvae/models/fastai_xresnet1d101.pth"

s3 = boto3.client("s3")
obj = s3.get_object(Bucket="walkky-ml", Key=model_key)
data = obj["Body"].read()
print("S3 object MD5:", hashlib.md5(data).hexdigest())
print("Size (bytes):", len(data))

resp = s3.head_object(Bucket="walkky-ml", Key=model_key)
print(resp["LastModified"], resp["ETag"], resp.get("VersionId"))

S3 object MD5: 703de91008e21da032f838d980ebb6eb
Size (bytes): 22998039
2026-07-17 18:09:32+00:00 "703de91008e21da032f838d980ebb6eb" None


In [29]:
# Compare xresnet1d101 models 

model_key = "aruna-files/vqvae_12lead_d384_bg_cnn_4h5/vqvae/models/fastai_xresnet1d101.pth"

obj = s3.get_object(Bucket="walkky-ml", Key=model_key)
data = obj["Body"].read()
print("S3 object MD5:", hashlib.md5(data).hexdigest())
print("Size (bytes):", len(data))

resp = s3.head_object(Bucket="walkky-ml", Key=model_key)
print(resp["LastModified"], resp["ETag"], resp.get("VersionId"))

S3 object MD5: 703de91008e21da032f838d980ebb6eb
Size (bytes): 22998039
2026-07-15 19:23:13+00:00 "703de91008e21da032f838d980ebb6eb" None


In [32]:
import os
from datetime import datetime

# Compare xresnet1d101 models 

# Path to local model file
local_path = "./models/fastai_xresnet1d101.pth"

# Read the file contents
with open(local_path, "rb") as f:
    data = f.read()

print("Local file MD5:", hashlib.md5(data).hexdigest())
print("Size (bytes):", len(data))

# Get file metadata
stat_info = os.stat(local_path)
print("LastModified:", datetime.fromtimestamp(stat_info.st_mtime))
print("ETag (MD5):", hashlib.md5(data).hexdigest())  # S3 ETag is usually MD5 for non-multipart uploads


Local file MD5: 703de91008e21da032f838d980ebb6eb
Size (bytes): 22998039
LastModified: 2026-06-10 09:42:03.948191
ETag (MD5): 703de91008e21da032f838d980ebb6eb


In [47]:
# check singly-labeled sentences with xrestnet embeddings - these are not the same sizes, errors out

# Output key and prefixes
bucket_out = 'walkky-ml'
prefix_out_1 = "aruna-files/vqvae_12lead_d384_bg_cnn_4h5/vqvae/bert_finetuning"
#prefix_out_2 = "aruna-files/vqvae_12lead_d768_bg_cnn_4h5/vqvae/bert_finetuning"
prefix_out_2 = "aruna-files/vqvae_final_12lead_singlelabel/vqvae/bert_finetuning"

# Initialize the S3 client
s3_client = boto3.client('s3')
s3_fs = s3fs.S3FileSystem()

dataset_name = "cpsc2018"
use_frac = 1.0
for split_name in ["train"]:
    check_sentences(prefix_out_1, prefix_out_2, dataset_name, split_name, use_frac, num_shards=4)

ValueError: operands could not be broadcast together with shapes (6539,41) (6605,41) 